# RT-DETR Knowledge Distillation — Phase 2 Training

**Colab Pro+ · A100 40GB**

| Phase | Dataset | Epochs | Runs | Purpose |
|-------|---------|--------|------|---------|
| **2A** | COCO 30K | 36 | 18 | Ablation / method selection |
| **2D** | Full COCO 118K | 72 | ~8 | Final paper numbers |
| **2E** | Full COCO 118K | 72 | 3 seeds | Statistical reliability |

All checkpoints are saved directly to Drive. Sessions can drop and resume transparently.

## 1. Environment Setup

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR    = '/content/rt_detr'
DRIVE_BASE  = '/content/drive/MyDrive/rt_detr'
RUNS_2A     = f'{DRIVE_BASE}/runs_2a'
RUNS_2D     = f'{DRIVE_BASE}/runs_2d'
WEIGHTS_DIR = f'{DRIVE_BASE}/weights'
COCO_DRIVE  = f'{DRIVE_BASE}/coco'
COCO_LOCAL  = '/content/coco'

for d in [RUNS_2A, RUNS_2D, WEIGHTS_DIR, COCO_DRIVE]:
    os.makedirs(d, exist_ok=True)

# Clone or pull repo
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/umutonuryasar/RT-DETR {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}

# Dependencies
!pip install -q pycocotools tensorboard

# GPU check
!nvidia-smi
print('\nSetup complete.')

## 2. COCO Data

Run the appropriate cell once. After download, data is kept on Drive so it
survives session drops. Subsequent sessions just symlink.

In [ ]:
def setup_coco(source_drive_path, local_path):
    """Symlink COCO from Drive to local path for fast I/O."""
    if os.path.islink(local_path):
        print(f'Already symlinked: {local_path}')
        return
    if os.path.exists(source_drive_path):
        os.symlink(source_drive_path, local_path)
        print(f'Symlinked {source_drive_path} → {local_path}')
    else:
        print(f'NOT FOUND on Drive: {source_drive_path}')
        print('Download first — see cells below.')

setup_coco(COCO_DRIVE, COCO_LOCAL)

In [ ]:
# ── Download 30K subset (Phase 2A) ── Run once, skip if Drive copy exists
subset_train = f'{COCO_DRIVE}/train2017_30k'
if not os.path.exists(subset_train):
    !bash scripts/download_coco_subset.sh {COCO_DRIVE}
else:
    print('30K subset already on Drive.')

In [ ]:
# ── Download full COCO 118K (Phase 2D/2E) ── Run once, ~19GB, takes ~20 min
full_train = f'{COCO_DRIVE}/train2017'
if not os.path.exists(full_train):
    !bash scripts/download_coco_full.sh {COCO_DRIVE}
else:
    n = len([f for f in os.listdir(full_train) if f.endswith('.jpg')])
    print(f'Full COCO already on Drive: {n:,} train images.')

## 3. Teacher Weights

Download RT-DETR pretrained weights from PaddleDetection and place under
`WEIGHTS_DIR` on Drive. Expected filenames:
- `rtdetr_r50vd_2x_coco.pth` — R50 teacher (run08, run13, run14-17)
- `rtdetr_r34vd_2x_coco.pth` — R34 teacher (run12)

In [ ]:
TEACHER_WEIGHTS_R50 = f'{WEIGHTS_DIR}/rtdetr_r50vd_2x_coco.pth'
TEACHER_WEIGHTS_R34 = f'{WEIGHTS_DIR}/rtdetr_r34vd_2x_coco.pth'

print('R50 teacher:', '✓ found' if os.path.exists(TEACHER_WEIGHTS_R50) else '✗ missing')
print('R34 teacher:', '✓ found' if os.path.exists(TEACHER_WEIGHTS_R34) else '✗ missing')
print(f'\nPlace weights at: {WEIGHTS_DIR}/')

## 4. Run Registry

All 18 Phase 2A configurations defined in one place.
Phase 2D/2E configs are in Section 7.

In [ ]:
# ── Phase 2A: 30K subset, 36 epochs ──────────────────────────────────────────
PHASE_2A_COCO_TRAIN  = f'{COCO_LOCAL}/train2017_30k'
PHASE_2A_COCO_VAL    = f'{COCO_LOCAL}/val2017'
PHASE_2A_TRAIN_ANN   = f'{COCO_LOCAL}/annotations/instances_train2017_30k.json'
PHASE_2A_VAL_ANN     = f'{COCO_LOCAL}/annotations/instances_val2017.json'
PHASE_2A_EPOCHS      = 36
PHASE_2A_BATCH       = 16    # A100 40GB
PHASE_2A_IMG_SIZE    = 640

# teacher_cfg / teacher_weights keys:
#   None  → use defaults (R50)
#   str   → explicit override
RUN_REGISTRY = {
    'run00_baseline':            dict(kd_type='none',           kd_lambda=0.0, temperature=4, kd_cfg='',                                  teacher_cfg=None, teacher_weights=None),
    'run01_logit_l0.5_t2':       dict(kd_type='logit',          kd_lambda=0.5, temperature=2, kd_cfg='',                                  teacher_cfg=None, teacher_weights=None),
    'run02_logit_l0.5_t4':       dict(kd_type='logit',          kd_lambda=0.5, temperature=4, kd_cfg='',                                  teacher_cfg=None, teacher_weights=None),
    'run03_logit_l0.5_t8':       dict(kd_type='logit',          kd_lambda=0.5, temperature=8, kd_cfg='',                                  teacher_cfg=None, teacher_weights=None),
    'run04_logit_l1.0_t2':       dict(kd_type='logit',          kd_lambda=1.0, temperature=2, kd_cfg='',                                  teacher_cfg=None, teacher_weights=None),
    'run05_logit_l1.0_t4':       dict(kd_type='logit',          kd_lambda=1.0, temperature=4, kd_cfg='',                                  teacher_cfg=None, teacher_weights=None),
    'run06_logit_l1.0_t8':       dict(kd_type='logit',          kd_lambda=1.0, temperature=8, kd_cfg='',                                  teacher_cfg=None, teacher_weights=None),
    'run07_feature_l0.5':        dict(kd_type='feature',        kd_lambda=0.5, temperature=4, kd_cfg='',                                  teacher_cfg=None, teacher_weights=None),
    'run08_feature_l1.0':        dict(kd_type='feature',        kd_lambda=1.0, temperature=4, kd_cfg='',                                  teacher_cfg=None, teacher_weights=None),
    'run09_combined_l1.0_t4':    dict(kd_type='combined',       kd_lambda=1.0, temperature=4, kd_cfg='configs/kd/combined_kd.yml',        teacher_cfg=None, teacher_weights=None),
    'run10_encoder_only_l1.0':   dict(kd_type='feature',        kd_lambda=1.0, temperature=4, kd_cfg='configs/kd/encoder_only_kd.yml',    teacher_cfg=None, teacher_weights=None),
    'run11_attention_only_l1.0': dict(kd_type='feature',        kd_lambda=1.0, temperature=4, kd_cfg='configs/kd/attention_only_kd.yml',  teacher_cfg=None, teacher_weights=None),
    'run12_feature_teacher_r34': dict(kd_type='feature',        kd_lambda=1.0, temperature=4, kd_cfg='',                                  teacher_cfg='configs/rtdetr_r34vd_coco.yml', teacher_weights='R34'),
    'run13_feature_teacher_r50': dict(kd_type='feature',        kd_lambda=1.0, temperature=4, kd_cfg='',                                  teacher_cfg=None, teacher_weights=None),
    'run14_cwd_l1.0':            dict(kd_type='cwd',            kd_lambda=1.0, temperature=4, kd_cfg='configs/kd/cwd_kd.yml',             teacher_cfg=None, teacher_weights=None),
    'run15_mgd_l1.0':            dict(kd_type='mgd',            kd_lambda=1.0, temperature=4, kd_cfg='configs/kd/mgd_kd.yml',             teacher_cfg=None, teacher_weights=None),
    'run16_query_kd_l1.0':       dict(kd_type='query',          kd_lambda=1.0, temperature=4, kd_cfg='configs/kd/query_kd.yml',           teacher_cfg=None, teacher_weights=None),
    'run17_stage_adaptive_l1.0': dict(kd_type='stage_adaptive', kd_lambda=1.0, temperature=4, kd_cfg='configs/kd/stage_adaptive_kd.yml',  teacher_cfg=None, teacher_weights=None),
}

STUDENT_CFG      = 'configs/rtdetr_r18vd_coco.yml'
DEFAULT_TEACHER_CFG = 'configs/rtdetr_r50vd_coco.yml'

print(f'Registry loaded: {len(RUN_REGISTRY)} runs')

## 5. Progress Tracker

In [ ]:
import re

def extract_map(eval_log_path):
    """Parse mAP@[.5:.95] from eval.log."""
    if not os.path.exists(eval_log_path):
        return None
    with open(eval_log_path) as f:
        content = f.read()
    # Matches: Average Precision  (AP) @[ IoU=0.50:0.95 | ... ] = 0.XXX
    m = re.search(r'IoU=0\.50:0\.95.*?=\s*([0-9.]+)', content)
    return float(m.group(1)) if m else None

def extract_fps(fps_log_path):
    """Parse FPS from fps.log."""
    if not os.path.exists(fps_log_path):
        return None
    with open(fps_log_path) as f:
        content = f.read()
    m = re.search(r'FPS[:\s]+([0-9.]+)', content, re.IGNORECASE)
    return float(m.group(1)) if m else None

def show_progress(output_root=None):
    """Print a status table for all 18 Phase 2A runs."""
    root = output_root or RUNS_2A
    rows = []
    for tag in RUN_REGISTRY:
        out_dir = f'{root}/{tag}'
        best = f'{out_dir}/checkpoint_best.pth'
        last = f'{out_dir}/checkpoint_last.pth'
        log  = f'{out_dir}/train.log'

        if os.path.exists(best):
            mAP = extract_map(f'{out_dir}/eval.log')
            fps = extract_fps(f'{out_dir}/fps.log')
            status = 'DONE'
            mAP_str = f'{mAP:.3f}' if mAP else '—'
            fps_str = f'{fps:.1f}' if fps else '—'
        elif os.path.exists(last):
            status = 'PARTIAL'
            mAP_str = fps_str = '—'
        else:
            status = 'pending'
            mAP_str = fps_str = '—'

        rows.append((tag, status, mAP_str, fps_str))

    done  = sum(1 for _, s, _, _ in rows if s == 'DONE')
    part  = sum(1 for _, s, _, _ in rows if s == 'PARTIAL')
    total = len(rows)

    print(f'{"Run":<35} {"Status":<10} {"mAP":>8} {"FPS":>8}')
    print('-' * 65)
    for tag, status, mAP_str, fps_str in rows:
        sym = '✓' if status == 'DONE' else ('~' if status == 'PARTIAL' else ' ')
        print(f'{sym} {tag:<33} {status:<10} {mAP_str:>8} {fps_str:>8}')
    print('-' * 65)
    print(f'Completed: {done}/{total}  |  In-progress: {part}  |  Pending: {total - done - part}')

show_progress()

## 6. Run Training

**Instructions:**
1. Set `RUN_TAG` in the cell below to the run you want to launch.
2. Run the config cell (auto-detects resume).
3. Run the training cell.
4. Run the evaluation cell.

Runs are saved to Drive — if the session drops, re-run from step 2 and training
will resume from `checkpoint_last.pth` automatically.

In [ ]:
# ── SET THIS before each run ────────────────────────────────────────────────
RUN_TAG = 'run00_baseline'   # ← change to any key in RUN_REGISTRY
# ────────────────────────────────────────────────────────────────────────────

assert RUN_TAG in RUN_REGISTRY, f'Unknown run: {RUN_TAG}. Check RUN_REGISTRY.'
cfg = RUN_REGISTRY[RUN_TAG]

OUTPUT_DIR = f'{RUNS_2A}/{RUN_TAG}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Resolve teacher paths
teacher_cfg = cfg.get('teacher_cfg') or DEFAULT_TEACHER_CFG
if cfg.get('teacher_weights') == 'R34':
    teacher_weights = TEACHER_WEIGHTS_R34
elif cfg['kd_type'] == 'none':
    teacher_weights = ''
else:
    teacher_weights = TEACHER_WEIGHTS_R50

# Auto-resume
last_ckpt = f'{OUTPUT_DIR}/checkpoint_last.pth'
resume_flag = f'--resume {last_ckpt}' if os.path.exists(last_ckpt) else ''

print(f'Run tag     : {RUN_TAG}')
print(f'KD type     : {cfg["kd_type"]}')
print(f'KD lambda   : {cfg["kd_lambda"]}')
print(f'Temperature : {cfg["temperature"]}')
print(f'KD config   : {cfg["kd_cfg"] or "<none>"}')
print(f'Teacher cfg : {teacher_cfg}')
print(f'Teacher wts : {teacher_weights or "<none>"}')
print(f'Output dir  : {OUTPUT_DIR}')
print(f'Resume      : {resume_flag or "<fresh start>"}')

In [ ]:
# ── Build CLI flags ──────────────────────────────────────────────────────────
flags = [
    f'--student-cfg {STUDENT_CFG}',
    f'--teacher-cfg {teacher_cfg}',
    f'--kd-type {cfg["kd_type"]}',
    f'--kd-lambda {cfg["kd_lambda"]}',
    f'--temperature {cfg["temperature"]}',
    f'--epochs {PHASE_2A_EPOCHS}',
    f'--batch-size {PHASE_2A_BATCH}',
    f'--img-size {PHASE_2A_IMG_SIZE}',
    f'--output-dir {OUTPUT_DIR}',
    f'--coco-train {PHASE_2A_COCO_TRAIN}',
    f'--coco-val {PHASE_2A_COCO_VAL}',
    f'--train-ann {PHASE_2A_TRAIN_ANN}',
    f'--val-ann {PHASE_2A_VAL_ANN}',
    '--use-amp',
    '--save-attn-maps',   # save attention maps for visualization
]

if teacher_weights:
    flags.append(f'--teacher-weights {teacher_weights}')
if cfg['kd_cfg']:
    flags.append(f'--kd-cfg {cfg["kd_cfg"]}')
if resume_flag:
    flags.append(resume_flag)

cmd = 'python tools/train_kd.py ' + ' '.join(flags)
print(cmd)
print()

# ── Launch ───────────────────────────────────────────────────────────────────
!{cmd} 2>&1 | tee {OUTPUT_DIR}/train.log

In [ ]:
# ── Evaluation ───────────────────────────────────────────────────────────────
!python tools/eval.py \
    --cfg {STUDENT_CFG} \
    --weights {OUTPUT_DIR}/checkpoint_best.pth \
    --coco-val {PHASE_2A_COCO_VAL} \
    --val-ann {PHASE_2A_VAL_ANN} \
    --img-size {PHASE_2A_IMG_SIZE} \
    2>&1 | tee {OUTPUT_DIR}/eval.log

# ── FPS benchmark ────────────────────────────────────────────────────────────
!python tools/benchmark_fps.py \
    --cfg {STUDENT_CFG} \
    --weights {OUTPUT_DIR}/checkpoint_best.pth \
    --input-size {PHASE_2A_IMG_SIZE} \
    --warmup 50 \
    --iters 500 \
    --device cuda \
    2>&1 | tee {OUTPUT_DIR}/fps.log

# ── Show updated progress ─────────────────────────────────────────────────────
print()
show_progress()

## 7. Results Summary

In [ ]:
def results_table(output_root=None):
    """Print a paper-ready summary table of completed runs."""
    root = output_root or RUNS_2A
    baseline_map = None
    rows = []

    for tag, cfg in RUN_REGISTRY.items():
        out_dir = f'{root}/{tag}'
        mAP = extract_map(f'{out_dir}/eval.log')
        fps = extract_fps(f'{out_dir}/fps.log')
        if mAP is None:
            continue
        if tag == 'run00_baseline':
            baseline_map = mAP
        rows.append((tag, cfg['kd_type'], cfg['kd_lambda'], cfg['temperature'], mAP, fps))

    if not rows:
        print('No completed runs yet.')
        return

    print(f'{"Run":<35} {"KD type":<16} {"λ":>5} {"T":>3} {"mAP":>7} {"ΔMAP":>7} {"FPS":>7}')
    print('-' * 82)
    for tag, kd_type, lam, T, mAP, fps in sorted(rows, key=lambda x: -x[4]):
        delta = f'{mAP - baseline_map:+.3f}' if baseline_map is not None else '—'
        fps_str = f'{fps:.1f}' if fps else '—'
        T_str = str(T) if kd_type not in ('none', 'feature', 'cwd', 'mgd', 'query') else '—'
        print(f'{tag:<35} {kd_type:<16} {lam:>5} {T_str:>3} {mAP:>7.3f} {delta:>7} {fps_str:>7}')

results_table()

## 8. TensorBoard

Monitor live training or compare completed runs.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {RUNS_2A}

## 9. Phase 2D — Final Paper Runs

**Run this section only after Phase 2A is complete.**

Select the top configurations from the results table above, then run them on
full COCO for 72 epochs with 3 seeds each.

Edit `FINAL_RUNS` below to reflect Phase 2A winners.

In [ ]:
# ── Phase 2D / 2E config ─────────────────────────────────────────────────────
PHASE_2D_COCO_TRAIN = f'{COCO_LOCAL}/train2017'
PHASE_2D_COCO_VAL   = f'{COCO_LOCAL}/val2017'
PHASE_2D_TRAIN_ANN  = f'{COCO_LOCAL}/annotations/instances_train2017.json'
PHASE_2D_VAL_ANN    = f'{COCO_LOCAL}/annotations/instances_val2017.json'
PHASE_2D_EPOCHS     = 72
PHASE_2D_BATCH      = 16
PHASE_2D_IMG_SIZE   = 640
PHASE_2D_SEEDS      = [42, 1337, 2025]

# ── Edit this after Phase 2A ─────────────────────────────────────────────────
# Always include: baseline, CWD, MGD + best from each KD family
FINAL_RUNS = [
    'run00_baseline',
    'run08_feature_l1.0',      # best feature-KD (adjust if run07 wins)
    'run14_cwd_l1.0',
    'run15_mgd_l1.0',
    # TODO: add best logit run after Phase 2A
    # 'run05_logit_l1.0_t4',
    # TODO: add best novel run after Phase 2A
    # 'run16_query_kd_l1.0',
]

print('Phase 2D runs:', FINAL_RUNS)
print(f'Seeds: {PHASE_2D_SEEDS}')
print(f'Total training jobs: {len(FINAL_RUNS) * len(PHASE_2D_SEEDS)}')

In [ ]:
# ── Launch one Phase 2D run ───────────────────────────────────────────────────
# Set these before each run:
P2D_RUN_TAG = 'run00_baseline'
P2D_SEED    = 42

assert P2D_RUN_TAG in FINAL_RUNS, f'{P2D_RUN_TAG} not in FINAL_RUNS'
cfg_2d = RUN_REGISTRY[P2D_RUN_TAG]

out_tag = f'{P2D_RUN_TAG}_seed{P2D_SEED}'
output_dir_2d = f'{RUNS_2D}/{out_tag}'
os.makedirs(output_dir_2d, exist_ok=True)

teacher_cfg_2d = cfg_2d.get('teacher_cfg') or DEFAULT_TEACHER_CFG
if cfg_2d.get('teacher_weights') == 'R34':
    tw_2d = TEACHER_WEIGHTS_R34
elif cfg_2d['kd_type'] == 'none':
    tw_2d = ''
else:
    tw_2d = TEACHER_WEIGHTS_R50

last_2d = f'{output_dir_2d}/checkpoint_last.pth'
resume_2d = f'--resume {last_2d}' if os.path.exists(last_2d) else ''

flags_2d = [
    f'--student-cfg {STUDENT_CFG}',
    f'--teacher-cfg {teacher_cfg_2d}',
    f'--kd-type {cfg_2d["kd_type"]}',
    f'--kd-lambda {cfg_2d["kd_lambda"]}',
    f'--temperature {cfg_2d["temperature"]}',
    f'--epochs {PHASE_2D_EPOCHS}',
    f'--batch-size {PHASE_2D_BATCH}',
    f'--img-size {PHASE_2D_IMG_SIZE}',
    f'--output-dir {output_dir_2d}',
    f'--coco-train {PHASE_2D_COCO_TRAIN}',
    f'--coco-val {PHASE_2D_COCO_VAL}',
    f'--train-ann {PHASE_2D_TRAIN_ANN}',
    f'--val-ann {PHASE_2D_VAL_ANN}',
    f'--seed {P2D_SEED}',
    '--use-amp',
    '--save-attn-maps',
]
if tw_2d:
    flags_2d.append(f'--teacher-weights {tw_2d}')
if cfg_2d['kd_cfg']:
    flags_2d.append(f'--kd-cfg {cfg_2d["kd_cfg"]}')
if resume_2d:
    flags_2d.append(resume_2d)

cmd_2d = 'python tools/train_kd.py ' + ' '.join(flags_2d)
print(f'Launching: {out_tag}')
print(cmd_2d)
print()
!{cmd_2d} 2>&1 | tee {output_dir_2d}/train.log

In [ ]:
# ── Evaluate Phase 2D run ────────────────────────────────────────────────────
!python tools/eval.py \
    --cfg {STUDENT_CFG} \
    --weights {output_dir_2d}/checkpoint_best.pth \
    --coco-val {PHASE_2D_COCO_VAL} \
    --val-ann {PHASE_2D_VAL_ANN} \
    --img-size {PHASE_2D_IMG_SIZE} \
    2>&1 | tee {output_dir_2d}/eval.log

!python tools/benchmark_fps.py \
    --cfg {STUDENT_CFG} \
    --weights {output_dir_2d}/checkpoint_best.pth \
    --input-size {PHASE_2D_IMG_SIZE} \
    --warmup 50 --iters 500 --device cuda \
    2>&1 | tee {output_dir_2d}/fps.log

In [ ]:
import numpy as np

def phase2d_results():
    """Aggregate Phase 2D results: mean ± std across seeds per run."""
    print(f'{"Run":<35} {"mAP mean":>10} {"mAP std":>10} {"FPS":>8}')
    print('-' * 67)

    for tag in FINAL_RUNS:
        maps, fps_vals = [], []
        for seed in PHASE_2D_SEEDS:
            out_dir = f'{RUNS_2D}/{tag}_seed{seed}'
            m = extract_map(f'{out_dir}/eval.log')
            f = extract_fps(f'{out_dir}/fps.log')
            if m is not None:
                maps.append(m)
            if f is not None:
                fps_vals.append(f)

        if maps:
            mean_map = np.mean(maps)
            std_map  = np.std(maps)
            fps_str  = f'{np.mean(fps_vals):.1f}' if fps_vals else '—'
            print(f'{tag:<35} {mean_map:>10.3f} {std_map:>10.4f} {fps_str:>8}  (n={len(maps)})')
        else:
            print(f'{tag:<35} {"—":>10} {"—":>10} {"—":>8}')

phase2d_results()